In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


CALCULATION PARAMETERS FOR COLOR NORMALIZATION

In [ ]:
import cv2
import numpy as np
import os
from tqdm import tqdm  # For the progress bar


# Insert the exact path of the folder containing the IMAGES
TARGET_IMG_DIR = '...'

# Insert the exact path of the folder where the ROI are saved (obtained using the function ROI_detection)
TARGET_MASK_DIR = '...'

pixel_count = 0
channel_sum = np.zeros(3, dtype=np.float64)       # Sum [L, A, B]
channel_sq_sum = np.zeros(3, dtype=np.float64)    # Sum of squares [L, A, B]

# Check if the directories exist
if not os.path.exists(TARGET_IMG_DIR) or not os.path.exists(TARGET_MASK_DIR):
    print(f" Error: One of the specified folders does not exist.")
    print(f"Img: {TARGET_IMG_DIR}")
    print(f"Msk: {TARGET_MASK_DIR}")
else:
    # Get the list of files from the images folder
    file_list = [f for f in os.listdir(TARGET_IMG_DIR) if f.lower().endswith(('.png', '.jpg', '.tif', '.bmp'))]

    print(f" Analyzing folder: {len(file_list)} images found.")

    for filename in tqdm(file_list):
        img_path = os.path.join(TARGET_IMG_DIR, filename)
        mask_path = os.path.join(TARGET_MASK_DIR, filename)

        # 1. Load Image and Mask
        # The mask must exist, otherwise skip the iteration
        if not os.path.exists(mask_path):
            # Handle different extensions here if necessary (e.g., .png vs .jpg)
            continue

        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            continue

        # 2. Convert to LAB color space
        img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype("float32")

        # 3. Extract valid pixels
        # Select only the pixels where the mask is greater than 0
        valid_pixels = img_lab[mask > 0]

        # Skip if the mask is completely black or empty
        if valid_pixels.size == 0:
            continue

        # 4. Update Accumulators
        n_pixels = valid_pixels.shape[0]

        pixel_count += n_pixels
        channel_sum += np.sum(valid_pixels, axis=0)
        channel_sq_sum += np.sum(valid_pixels ** 2, axis=0)

    # ---- FINAL CALCULATION ----
    if pixel_count > 0:
        # Mean = Sum / N
        global_mean = channel_sum / pixel_count

        # StdDev = Sqrt( (SumSquares / N) - Mean^2 )
        # Variance = E[X^2] - (E[X])^2
        global_variance = (channel_sq_sum / pixel_count) - (global_mean ** 2)

        # Ensure variance is not negative due to precision errors
        global_variance = np.maximum(global_variance, 0)

        global_std = np.sqrt(global_variance)

        print("\n CALCULATION COMPLETED!")
        print(f"Total pixels analyzed: {pixel_count}")
        print("-" * 30)
        print("Copy these values into your function:")
        print(f"TARGET_MEAN = [{global_mean[0]:.4f}, {global_mean[1]:.4f}, {global_mean[2]:.4f}]")
        print(f"TARGET_STD  = [{global_std[0]:.4f}, {global_std[1]:.4f}, {global_std[2]:.4f}]")
        print("-" * 30)

    else:
        print(" No valid pixels found. Check paths and ensure masks are not all black.")